In [ ]:
import numpy as np, os, time
import tensorflow as tf
from tensorflow import keras
from argparse import Namespace
from tensorflow.keras.callbacks import ModelCheckpoint
from tensorflow.keras.optimizers import Adam
import holoviews as hv
from holoviews import opts
hv.extension('bokeh')
opts.defaults( opts.Curve(width=1000, height=600, line_width=1) )
import tmodel
use_current = True

if use_current:
    args: Namespace = tmodel.load_args()
    signal_index=args.signal
    feature_type=args.feature_type
else:
    signal_index=2
    feature_type=1
    args: Namespace = tmodel.load_args(signal_index,feature_type)

In [ ]:
data=tmodel.get_demo_data()
signals = data['signals']
times = data['times']
T: np.ndarray = times[signal_index].copy()
X: np.ndarray = tmodel.get_features( T, feature_type, args )
Y: np.ndarray = signals[signal_index]
F = np.arange(X.shape[1])
print( f"X.shape={X.shape} Y.shape={Y.shape} F.shape={F.shape}")

strategy = tf.distribute.MirroredStrategy([f"GPU:{i}" for i in args.devices])
print(f"Number of devices: {strategy.num_replicas_in_sync}")
with strategy.scope():
    model = tmodel.create_streams_model( X.shape[1], 0.0, n_streams=args.nstreams )
    model.compile( optimizer=tf.keras.optimizers.Adam( learning_rate=0.01 ), loss=args.loss )

latest_ckp_file = tmodel.get_ckp_file( args, "latest" )
assert os.path.exists(latest_ckp_file), f"Checkpint file '{latest_ckp_file}' not found."
print( f"Loading checkpoint from '{latest_ckp_file}'")
model.load_weights(latest_ckp_file)

A, P = tmodel.get_masked_attribution( model, X )
Tt = T[:P.shape[1]]

In [ ]:
curve_dict = { fi: hv.Curve((Tt, P[fi]), 'Time', 'Amplitude')  for fi in range(P.shape[0]) }
kdims = [ hv.Dimension(('feature', 'Feature'), default=0) ]
holomap = hv.HoloMap(curve_dict, kdims=kdims)
holomap.opts(opts.Curve(width=1200, title="Masked Feature Results"))